# Practice 05 Additional — Experiment Tracking

**Goal:** keep every training run on record — its settings, its scores and its model file — with MLflow Tracking, and let Optuna write that record for us.

| Section | What is recorded | How |
|:---|:---|:---|
| 1 | one training, by hand | `mlflow.start_run`, `log_params`, `log_metrics`, `log_artifact` |
| 2 | every Optuna trial | `MLflowCallback` |
| 3 | the record, read back | `mlflow.search_runs`, `mlflow ui` |
| 4 | the same idea, hosted | Weights & Biases |

- Data → `fetch_openml('vehicle')` as in Practice 05 (846 silhouettes, 18 features, four classes), split once; a pipeline with a scaler, scored by 5-fold `cv`
- Install → the first code cell; about one minute in Colab, once per session

---
## Setup and Data

In [1]:
import os
import subprocess
import sys
os.environ['MLFLOW_DISABLE_AGENT_HINT'] = '1'   # MLflow 3 prints a hint for AI agents on import

try:
    import optuna
    import mlflow
    import optuna_integration
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'optuna', 'mlflow', 'optuna-integration[mlflow]'], check=True)
    import optuna
    import mlflow
    import optuna_integration

print(f'optuna {optuna.__version__}, mlflow {mlflow.__version__}')

optuna 4.9.0, mlflow 3.17.2


In [2]:
import os
os.environ['OMP_NUM_THREADS'] = '1'   # must be set before sklearn is imported

import time
import warnings
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
np.random.seed(42)

In [3]:
vehicle = fetch_openml('vehicle', version=1, as_frame=True)
X = vehicle.data
y = vehicle.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f'train   {X_train.shape}')
print(f'test    {X_test.shape}')

train   (676, 18)
test    (170, 18)


---
# 1. One Run by Hand

- A score in a variable → gone with the kernel, and nothing links it to the settings that produced it
- MLflow Tracking → a **run** per training: its **parameters**, **metrics** and **artifacts** (files), grouped into **experiments**
- Below: the record is the SQLite file `mlflow.db`; no server, no account

Documentation: <https://mlflow.org/docs/latest/ml/tracking/>

### The Baseline SVC, Measured Four Ways

- Test accuracy, fit seconds, predict seconds, model size → the same four numbers for every model that follows
- The model file saved with `joblib` → logged as an artifact into `mlruns/` next to the database, so the run keeps the model, not only its score

In [4]:
import logging
import joblib
import mlflow
logging.getLogger('mlflow').setLevel(logging.WARNING)   # MLflow's INFO lines (database setup, hints) are noise here

if os.path.exists('mlflow.db'):
    os.remove('mlflow.db')                            # start the record from scratch on every run of this notebook

mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('vehicle')

model_baseline = Pipeline([('scaler', StandardScaler()),
                           ('model',  SVC(random_state=42))])

start = time.perf_counter()
model_baseline.fit(X_train, y_train)
fit_seconds_baseline = time.perf_counter() - start

start = time.perf_counter()
y_pred = model_baseline.predict(X_test)
predict_seconds_baseline = time.perf_counter() - start

acc_test_baseline = accuracy_score(y_test, y_pred)
joblib.dump(model_baseline, 'baseline_svc.joblib')
size_mb_baseline = os.path.getsize('baseline_svc.joblib') / 1e6

with mlflow.start_run(run_name='baseline_svc'):
    mlflow.log_params({'family': 'SVC', 'C': 1.0, 'gamma': 'scale'})
    mlflow.log_metrics({'test_accuracy': acc_test_baseline,
                        'fit_seconds': fit_seconds_baseline,
                        'predict_seconds': predict_seconds_baseline,
                        'size_mb': size_mb_baseline})
    mlflow.log_artifact('baseline_svc.joblib')

print(f'test accuracy   : {acc_test_baseline:.4f}')
print(f'fit seconds     : {fit_seconds_baseline:.3f}')
print(f'predict seconds : {predict_seconds_baseline:.3f}')
print(f'size MB         : {size_mb_baseline:.2f}')

test accuracy   : 0.7647
fit seconds     : 0.008
predict seconds : 0.004
size MB         : 0.10


### What Landed in the Record

- `mlflow.search_runs` → the runs of an experiment as a `DataFrame`: one row per run, columns `params.…` and `metrics.…`

In [5]:
runs = mlflow.search_runs(experiment_names=['vehicle'])
runs[['tags.mlflow.runName', 'params.family', 'params.C', 'params.gamma',
      'metrics.test_accuracy', 'metrics.fit_seconds', 'metrics.size_mb']]

,tags.mlflow.runName,params.family,params.C,params.gamma,metrics.test_accuracy,metrics.fit_seconds,metrics.size_mb
0,baseline_svc,SVC,1.0,scale,0.764706,0.008025,0.100254


---
# 2. Every Trial as a Run — `MLflowCallback`

- Logging trials by hand means one block like the above per trial → the callback does it: one run per trial, parameters and value
- The study below searches the **model family** as well → its branch decides which hyperparameters are asked next
- `study_name` → the callback names its MLflow experiment after it

In [6]:
import optuna
from optuna.integration.mlflow import MLflowCallback
optuna.logging.set_verbosity(optuna.logging.WARNING)


def build_model(trial):                        # the family first, then only that family's hyperparameters
    family = trial.suggest_categorical(
        'family', ['LogisticRegression', 'KNeighborsClassifier', 'SVC', 'RandomForestClassifier'])

    if family == 'LogisticRegression':
        C = trial.suggest_float('logistic_C', 1e-2, 1e2, log=True)
        model = LogisticRegression(C=C, max_iter=2000, random_state=42)
    elif family == 'KNeighborsClassifier':
        n_neighbors = trial.suggest_int('knn_n_neighbors', 1, 15)
        model = KNeighborsClassifier(n_neighbors=n_neighbors)
    elif family == 'SVC':
        C = trial.suggest_float('svc_C', 1e-1, 1e3, log=True)
        gamma = trial.suggest_float('svc_gamma', 1e-4, 1e0, log=True)
        model = SVC(C=C, gamma=gamma, random_state=42)
    else:
        n_estimators = trial.suggest_int('rf_n_estimators', 50, 300, step=50)
        max_depth = trial.suggest_categorical('rf_max_depth', [None, 10, 20])
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)

    return Pipeline([('scaler', StandardScaler()), ('model', model)])


def objective(trial):
    candidate = build_model(trial)
    scores = cross_val_score(candidate, X_train, y_train, cv=cv, n_jobs=1)   # one process: no worker start-up while the SQLite record is open
    return np.mean(scores)


mlflow_callback = MLflowCallback(tracking_uri=mlflow.get_tracking_uri(), metric_name='cv_accuracy')

start = time.perf_counter()
study = optuna.create_study(study_name='vehicle_family', direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=60, callbacks=[mlflow_callback])
search_seconds = time.perf_counter() - start
acc_val_optuna = study.best_value

print('trials     :', len(study.trials))
print('best params:', study.best_params)
print(f'best CV    : {acc_val_optuna:.4f}')
print(f'seconds    : {search_seconds:.1f}')

trials     : 60
best params: {'family': 'SVC', 'svc_C': 420.26281761141377, 'svc_gamma': 0.015270096624184128}
best CV    : 0.8565
seconds    : 8.9


### Sixty Runs Without a Single `log_` Call

- The callback named its experiment after the study → `'vehicle_family'`; `runName` is the trial number
- Sorted by `cv_accuracy` → the top rows; `None` in a column means that trial never asked for the parameter
- The top row, trial 55, is `best_params`; the rows below it differ in the third decimal → differences a 60-trial search cannot tell apart

In [7]:
runs = mlflow.search_runs(experiment_names=['vehicle_family'], order_by=['metrics.cv_accuracy DESC'])

print('runs :', len(runs))
runs[['tags.mlflow.runName', 'params.family', 'params.svc_C', 'params.svc_gamma',
      'params.knn_n_neighbors', 'metrics.cv_accuracy']].head(10)

runs : 60


,tags.mlflow.runName,params.family,params.svc_C,params.svc_gamma,params.knn_n_neighbors,metrics.cv_accuracy
0,55,SVC,420.26281761141377,0.015270096624184128,None,0.856471
1,44,SVC,293.35278167910815,0.012736084462364542,None,0.849096
2,30,SVC,274.53612584523,0.011328788204854001,None,0.849085
3,31,SVC,271.27749487459414,0.010599956807711461,None,0.847614
4,38,SVC,76.71160699263012,0.026313124203613227,None,0.844608
5,52,SVC,117.4880099733885,0.015016642282075441,None,0.843170
6,32,SVC,159.39848749593855,0.0130086067257893,None,0.843170
7,23,SVC,205.1662550626444,0.011075043004613877,None,0.843170
8,33,SVC,135.83598980004618,0.02654823875390646,None,0.843126
9,41,SVC,60.517836142708816,0.025197054625499517,None,0.840207


### Budget per Family

- `study.trials` → the same 60 trials counted per family: where TPE spent its budget

In [8]:
print('family                   trials   best CV')
for family in ['LogisticRegression', 'KNeighborsClassifier', 'SVC', 'RandomForestClassifier']:
    family_values = []
    for trial in study.trials:
        if trial.params['family'] == family:
            family_values.append(trial.value)
    n_trials = len(family_values)
    best_value = max(family_values)
    print(f'{family:24s} {n_trials:4d}    {best_value:.4f}')

family                   trials   best CV
LogisticRegression          7    0.7914
KNeighborsClassifier        7    0.6938
SVC                        40    0.8565
RandomForestClassifier      6    0.7604


### The Winner, Logged Like the Baseline

- `FixedTrial(best_params)` → `build_model` returns the winning pipeline; fitted here, measured the same four ways
- `set_experiment('vehicle')` again → the callback had moved the active experiment to `'vehicle_family'`
- Search time counts as training time → the price of the search stays on the record

In [9]:
model_optuna = build_model(optuna.trial.FixedTrial(study.best_params))

start = time.perf_counter()
model_optuna.fit(X_train, y_train)
fit_seconds_optuna = time.perf_counter() - start + search_seconds

start = time.perf_counter()
y_pred = model_optuna.predict(X_test)
predict_seconds_optuna = time.perf_counter() - start

acc_test_optuna = accuracy_score(y_test, y_pred)
joblib.dump(model_optuna, 'optuna_best.joblib')
size_mb_optuna = os.path.getsize('optuna_best.joblib') / 1e6

mlflow.set_experiment('vehicle')
with mlflow.start_run(run_name='optuna_best'):
    mlflow.log_params(study.best_params)
    mlflow.log_metrics({'test_accuracy': acc_test_optuna,
                        'fit_seconds': fit_seconds_optuna,
                        'predict_seconds': predict_seconds_optuna,
                        'size_mb': size_mb_optuna})
    mlflow.log_artifact('optuna_best.joblib')

print(f'test accuracy   : {acc_test_optuna:.4f}')
print(f'fit seconds     : {fit_seconds_optuna:.1f}  (incl. {search_seconds:.1f} search)')
print(f'predict seconds : {predict_seconds_optuna:.3f}')
print(f'size MB         : {size_mb_optuna:.2f}')

test accuracy   : 0.8706
fit seconds     : 8.9  (incl. 8.9 search)
predict seconds : 0.002
size MB         : 0.06


---
# 3. Reading the Record

- The `'vehicle'` experiment now holds two runs → the comparison is on disk, settings included
- Restart the kernel and run only the next cell → the table is still there

In [10]:
import mlflow
mlflow.set_tracking_uri('sqlite:///mlflow.db')      # enough on its own after a kernel restart

runs = mlflow.search_runs(experiment_names=['vehicle'], order_by=['metrics.test_accuracy DESC'])
runs[['tags.mlflow.runName', 'params.family', 'metrics.test_accuracy',
      'metrics.fit_seconds', 'metrics.predict_seconds', 'metrics.size_mb']]

,tags.mlflow.runName,params.family,metrics.test_accuracy,metrics.fit_seconds,metrics.predict_seconds,metrics.size_mb
0,optuna_best,SVC,0.870588,8.880579,0.002236,0.058302
1,baseline_svc,SVC,0.764706,0.008025,0.003843,0.100254


### The Same Record in a Browser

- `mlflow ui` → a local web page over the same file: the experiments, every run, the artifacts to download

```
mlflow ui --backend-store-uri sqlite:///mlflow.db        # http://127.0.0.1:5000
```

- Colab opens no local ports → run it on your own machine, in the folder that holds `mlflow.db`

---
# 4. The Same Idea, Hosted — Weights & Biases

- Every tracker has the same two things → a run with parameters and metrics, and a dashboard that compares runs
- W&B keeps the record in its cloud → team sharing, live training curves, **sweeps**; needs an account, free for academic use

```python
import wandb

wandb.init(project='vehicle', config={'family': 'SVC', 'C': 1.0, 'gamma': 'scale'})
wandb.log({'test_accuracy': acc_test_baseline, 'fit_seconds': fit_seconds_baseline})
wandb.finish()
```

- Not run here → `init` opens, `log` writes, `finish` closes

---
## Summary

| | You write | It keeps |
|:---|:---|:---|
| `start_run` + `log_*` | what to log | one run: parameters, metrics, files |
| `MLflowCallback` | one argument | one run per trial |
| `search_runs`, `mlflow ui` | a query | the record, as a table or a page |
| Weights & Biases | `init`, `log`, `finish` | the same record, hosted |

- A score is useful only while it stays linked to its settings
- Search time belongs on the record → the winner's cost is part of the result